# 📊 Notebook 1 — Sentiment Analysis Ulasan UMKM

**Tujuan:** Fine-tune model IndoBERT-lite untuk mengklasifikasikan sentimen ulasan pelanggan UMKM ke dalam 3 kategori:
- 🔴 `negatif` (Sentiment_Score < -0.1)
- ⚪ `netral` (Sentiment_Score -0.1 s/d 0.1)
- 🟢 `positif` (Sentiment_Score > 0.1)

**Model:** `indobenchmark/indobert-lite-base-p1` (versi ringan IndoBERT, khusus Bahasa Indonesia)

**Dataset:** `UMKM-data/ulasan_umkm.csv` — 2000 baris ulasan berlabel

**Pendekatan:** Reproduce label yang sudah ada (Approach A) agar konsisten dengan logika bisnis UHP.

---

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Jalankan cell ini sekali untuk menginstall semua dependency
!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn accelerate -q

## 📚 Cell 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset

# Cek device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 📂 Cell 3 — Load & Eksplorasi Dataset

In [ ]:
# Load dataset ulasan
df = pd.read_csv('UMKM-data/ulasan_umkm.csv')

print('=== Info Dataset ===')
print(f'Total baris  : {len(df)}')
print(f'Kolom        : {list(df.columns)}')
print()
print('=== 5 Baris Pertama ===')
display(df.head())

print()
print('=== Cek Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Visualisasi distribusi label
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart distribusi label
label_counts = df['Sentiment_Label'].value_counts()
colors = ['#ef4444', '#6b7280', '#22c55e']
label_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribusi Sentiment Label', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(label_counts):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Distribusi Sentiment_Score
df['Sentiment_Score'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='#6366f1', edgecolor='white', linewidth=1.2
)
axes[1].set_title('Distribusi Sentiment Score (Numerik)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Jumlah')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('UMKM-data/distribusi_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan ke UMKM-data/distribusi_sentimen.png')

In [ ]:
# Contoh teks per label
print('=== Contoh Review per Label ===')
for label in ['positif', 'netral', 'negatif']:
    sample = df[df['Sentiment_Label'] == label]['Review_Text'].sample(2, random_state=42).values
    print(f'\n[{label.upper()}]')
    for i, text in enumerate(sample, 1):
        print(f'  {i}. {text[:100]}...' if len(text) > 100 else f'  {i}. {text}')

## 🔧 Cell 4 — Preprocessing & Label Encoding

In [ ]:
# Label mapping
LABEL2ID = {'negatif': 0, 'netral': 1, 'positif': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Bersihkan teks: hapus whitespace berlebih
df['Review_Text'] = df['Review_Text'].astype(str).str.strip()

# Encode label ke integer
df['label'] = df['Sentiment_Label'].map(LABEL2ID)

print('Label mapping:', LABEL2ID)
print()
print('Distribusi label (encoded):')
print(df['label'].value_counts().sort_index())
print()
print('Statistik panjang teks:')
df['text_len'] = df['Review_Text'].str.len()
print(df['text_len'].describe().round(1))

## ✂️ Cell 5 — Train / Test Split (80:20 Stratified)

In [ ]:
# Stratified split — menjaga proporsi label
train_df, test_df = train_test_split(
    df[['Review_Text', 'label', 'Sentiment_Label']],
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train set : {len(train_df)} baris')
print(f'Test set  : {len(test_df)} baris')
print()
print('Distribusi train set:')
print(train_df['Sentiment_Label'].value_counts())
print()
print('Distribusi test set:')
print(test_df['Sentiment_Label'].value_counts())

## 🤗 Cell 6 — Load Tokenizer IndoBERT-lite

In [ ]:
MODEL_NAME = 'indobenchmark/indobert-lite-base-p1'
MAX_LEN = 128  # Panjang max token (review UMKM umumnya pendek)

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer berhasil dimuat!')

# Cek contoh tokenisasi
contoh = 'Pelayanan cepat dan ramah, pesanan selalu tepat.'
tokens = tokenizer(contoh, return_tensors='pt', truncation=True, max_length=MAX_LEN)
print(f'\nContoh teks : "{contoh}"')
print(f'Input IDs   : {tokens["input_ids"].shape} tokens')
print(f'Token hasil : {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])}')

## 🗂️ Cell 7 — Buat Custom Dataset Class

In [ ]:
class UlasanDataset(Dataset):
    """Dataset class untuk ulasan UMKM."""
    
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts  = dataframe['Review_Text'].tolist()
        self.labels = dataframe['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Buat dataset object
train_dataset = UlasanDataset(train_df, tokenizer, MAX_LEN)
test_dataset  = UlasanDataset(test_df,  tokenizer, MAX_LEN)

print(f'Train dataset size : {len(train_dataset)}')
print(f'Test dataset size  : {len(test_dataset)}')
print()
# Cek satu sample
sample = train_dataset[0]
print('Contoh item dataset:')
for k, v in sample.items():
    print(f'  {k}: shape={v.shape}, dtype={v.dtype}')

## 🧠 Cell 8 — Load Model & Konfigurasi Training

In [ ]:
print(f'Loading model: {MODEL_NAME} ...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
model.to(device)

# Hitung jumlah parameter
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nModel berhasil dimuat!')
print(f'Total parameter    : {total_params:,}')
print(f'Trainable parameter: {trainable_params:,}')
print(f'Device             : {next(model.parameters()).device}')

In [ ]:
import os
os.makedirs('models/sentiment', exist_ok=True)

# Fungsi evaluasi untuk Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1_weighted': f1}

# Konfigurasi training
training_args = TrainingArguments(
    output_dir                  = 'models/sentiment/checkpoints',
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_weighted',
    greater_is_better           = True,
    logging_steps               = 50,
    save_total_limit            = 2,
    seed                        = SEED,
    report_to                   = 'none',   # Nonaktifkan W&B
    fp16                        = torch.cuda.is_available(),  # AMP jika ada GPU
)

print('Training arguments:')
print(f'  Epochs          : {training_args.num_train_epochs}')
print(f'  Batch size train: {training_args.per_device_train_batch_size}')
print(f'  Learning rate   : {training_args.learning_rate}')
print(f'  FP16 (GPU)      : {training_args.fp16}')

## 🏋️ Cell 9 — Training

In [ ]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Memulai fine-tuning IndoBERT-lite...')
print('=' * 50)
train_result = trainer.train()

print('\n=== Hasil Training ===')
print(f'Training loss akhir : {train_result.training_loss:.4f}')
print(f'Total steps         : {train_result.global_step}')
print(f'Waktu training      : {train_result.metrics["train_runtime"]:.1f} detik')

## 📈 Cell 10 — Evaluasi Model

In [ ]:
# Evaluasi pada test set
print('Mengevaluasi model pada test set...')
eval_results = trainer.evaluate(test_dataset)

print('\n=== Hasil Evaluasi ===')
print(f'Accuracy   : {eval_results["eval_accuracy"]:.4f} ({eval_results["eval_accuracy"]*100:.2f}%)')
print(f'F1 Weighted: {eval_results["eval_f1_weighted"]:.4f}')
print(f'Loss       : {eval_results["eval_loss"]:.4f}')

In [ ]:
# Prediksi pada test set untuk laporan detail
print('Mengambil prediksi pada test set...')
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

label_names = [ID2LABEL[i] for i in range(3)]

print('\n=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=label_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=label_names,
    yticklabels=label_names,
    linewidths=0.5,
    annot_kws={'size': 13, 'weight': 'bold'}
)
ax.set_title('Confusion Matrix — Sentiment Classifier\n(IndoBERT-lite, Test Set)', fontsize=13, fontweight='bold')
ax.set_xlabel('Prediksi', fontsize=11)
ax.set_ylabel('Aktual', fontsize=11)
plt.tight_layout()
plt.savefig('UMKM-data/confusion_matrix_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix disimpan ke UMKM-data/confusion_matrix_sentimen.png')

In [ ]:
# Plot training loss history
log_history = trainer.state.log_history
train_logs = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs  = [x for x in log_history if 'eval_loss' in x]

if train_logs and eval_logs:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Loss
    axes[0].plot([x['step'] for x in train_logs], [x['loss'] for x in train_logs],
                 marker='o', markersize=3, color='#6366f1', label='Train Loss')
    axes[0].plot([x['step'] for x in eval_logs], [x['eval_loss'] for x in eval_logs],
                 marker='s', markersize=5, color='#ef4444', label='Eval Loss')
    axes[0].set_title('Training & Eval Loss', fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy per epoch
    axes[1].plot([x['epoch'] for x in eval_logs], [x['eval_accuracy'] for x in eval_logs],
                 marker='D', markersize=7, color='#22c55e', linewidth=2)
    axes[1].set_title('Eval Accuracy per Epoch', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('UMKM-data/training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Training history disimpan ke UMKM-data/training_history.png')

## 💾 Cell 11 — Simpan Model

In [ ]:
SAVE_DIR = 'models/sentiment'

# Simpan model dan tokenizer
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f'Model disimpan ke: {SAVE_DIR}/')
print(f'  - config.json')
print(f'  - pytorch_model.bin / model.safetensors')
print(f'  - tokenizer files')

# Simpan label mapping
import json
label_info = {
    'label2id'     : LABEL2ID,
    'id2label'     : ID2LABEL,
    'model_name'   : MODEL_NAME,
    'max_length'   : MAX_LEN,
    'num_labels'   : 3,
    'eval_accuracy': round(eval_results['eval_accuracy'], 4),
    'eval_f1'      : round(eval_results['eval_f1_weighted'], 4),
    'score_mapping': {
        'positif': 'score > 0.1',
        'netral' : 'score -0.1 s/d 0.1',
        'negatif': 'score < -0.1'
    }
}
with open(f'{SAVE_DIR}/label_info.json', 'w') as f:
    json.dump(label_info, f, indent=2, ensure_ascii=False)
print(f'  - label_info.json')

## 🔮 Cell 12 — Inference: Uji Prediksi Manual

In [ ]:
def predict_sentiment(text, model, tokenizer, device, id2label=ID2LABEL):
    """Prediksi sentimen untuk satu teks."""
    model.eval()
    encoding = tokenizer(
        text,
        return_tensors='pt',
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        logits  = outputs.logits
        probs   = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
        pred_id = np.argmax(probs)

    return {
        'label'     : id2label[pred_id],
        'confidence': round(float(probs[pred_id]), 4),
        'scores'    : {
            id2label[i]: round(float(probs[i]), 4) for i in range(len(probs))
        }
    }


# Uji dengan beberapa contoh teks
test_texts = [
    'Pelayanan cepat dan ramah, pesanan selalu tepat. Sangat puas!',
    'Pengiriman cepat, admin komunikatif.',
    'Pelayanan standar, masih bisa ditingkatkan.',
    'Harga dan kualitas seimbang, pengalaman biasa saja.',
    'Pesanan sering terlambat saat jam sibuk.',
    'Respons admin lambat dan informasi kurang jelas. Kecewa sekali!',
    'Proses pembayaran sering bermasalah.',
]

print('=== Uji Prediksi Manual ===')
print(f'{"Teks":<55} {"Label":<10} {"Confidence":<12}')
print('-' * 80)
for text in test_texts:
    result = predict_sentiment(text, model, tokenizer, device)
    label_emoji = {'positif': 'POS', 'netral': 'NEU', 'negatif': 'NEG'}[result['label']]
    short_text = text[:52] + '...' if len(text) > 52 else text
    print(f'{short_text:<55} [{label_emoji}] {result["label"]:<10} {result["confidence"]*100:.1f}%')

## 🔄 Cell 13 — Generate Sentiment Score untuk Dataset Utama (Opsional)

> **Opsional:** Jalankan cell ini jika ingin mengupdate `Sentiment_Score` di CSV utama menggunakan model yang baru dilatih. Prosesnya bisa memakan waktu beberapa menit karena ada 150K baris.

In [ ]:
# Mapping label ke Sentiment_Score numerik
# Mengikuti skema score yang sama dengan dataset asli
LABEL_TO_SCORE = {
    'positif': 0.55,   # Rata-rata score positif di dataset asli
    'netral' : 0.0,    # Netral
    'negatif': -0.35   # Rata-rata score negatif di dataset asli
}

def batch_predict(texts, model, tokenizer, device, batch_size=64):
    """Prediksi batch untuk efisiensi."""
    model.eval()
    all_labels = []
    all_scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoding = tokenizer(
            batch,
            return_tensors='pt',
            max_length=MAX_LEN,
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            probs   = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            preds   = np.argmax(probs, axis=-1)

        for pred in preds:
            label = ID2LABEL[pred]
            all_labels.append(label)
            all_scores.append(LABEL_TO_SCORE[label])

        if (i // batch_size) % 10 == 0:
            print(f'  Progress: {i}/{len(texts)} ({i/len(texts)*100:.1f}%)', end='\r')

    return all_labels, all_scores


# Jalankan prediksi pada dataset utama
print('Memuat dataset utama...')
df_main = pd.read_csv('UMKM-data/synthetic_umkm_data_new_labels.csv')

print(f'Memprediksi sentimen untuk {len(df_main)} baris...')
texts = df_main['Review_Text'].astype(str).tolist()
pred_labels, pred_scores = batch_predict(texts, model, tokenizer, device)

# Update kolom
df_main['Sentiment_Score_Model'] = pred_scores
df_main['Sentiment_Label_Model'] = pred_labels

# Simpan versi baru
output_path = 'UMKM-data/synthetic_umkm_data_with_model_sentiment.csv'
df_main.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'\nSelesai! Disimpan ke: {output_path}')
print('\nDistribusi label hasil model:')
print(pd.Series(pred_labels).value_counts())

---

## ✅ Ringkasan

| Item | Detail |
|---|---|
| **Model** | `indobenchmark/indobert-lite-base-p1` (fine-tuned) |
| **Dataset** | `UMKM-data/ulasan_umkm.csv` — 2000 baris |
| **Split** | 80% train (1600), 20% test (400) — stratified |
| **Label** | negatif (0), netral (1), positif (2) |
| **Output model** | `models/sentiment/` |
| **Digunakan oleh** | Notebook 2 & `js/ml_engine.js` via `indobert_sentiment.onnx` |

**Langkah selanjutnya:** Lanjut ke **Notebook 2** untuk melatih model RF/XGBoost menggunakan fitur tabular termasuk `Sentiment_Score` sebagai salah satu fitur input.